# 02 · Модели и walk-forward кросс-валидация

Один обучающий цикл для всех моделей. Батч состоит из 10 торговых дней: для графовых моделей
(NSD, GCN, GAT) они склеиваются в один несвязный граф из 10 × 46 узлов, для табличных
(MLP, LSTM) это те же ~460 строк. У всех архитектур одинаковые батчи, оптимизатор и максимальный
бюджет (100 эпох); фактическое число эпох у каждого прогона своё из-за early stopping.

Результаты пишутся построчно в `results/runs.csv` и `results/predictions/`; уже посчитанные
тройки (конфигурация, фолд, сид) пропускаются, поэтому ноутбук можно прерывать и перезапускать.

In [1]:
import time
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import matthews_corrcoef, f1_score, accuracy_score
from torch_geometric.nn import GCNConv, GATConv

DATA, RESULTS = Path('data'), Path('results')
(RESULTS / 'predictions').mkdir(parents=True, exist_ok=True)
PRICE, FLAG, NEWS = slice(0, 6), 6, slice(7, None)  # 6 ценовых признаков, флаг has_news, 768 ruBERT
SEQ, K_DAYS, SEEDS = 20, 10, 5                       # окно LSTM, дней в батче, сидов на конфигурацию
MIN_TRAIN_ROWS = 120                                 # тикер без полугода истории в train фолда не размечается
torch.set_num_threads(1)                             # один поток → побитово воспроизводимый backward на CPU

def seed_all(s):
    torch.manual_seed(s); np.random.seed(s)

## Данные

Матрица признаков узлов и месячные графы собраны в `00_build_data.ipynb`. Граф месяца M построен
на конце месяца M−1, поэтому август–сентябрь 2022 (до первого графа) в моделирование не входят.
Дуги в `edge_index_corr.npz` лежат парами: сначала все прямые, затем все обратные; это позволяет
находить обратную дугу арифметикой, без словарей. Окно LSTM строится по скорректированным ценам
(`prices_adj.csv`) и заканчивается днём t включительно, как и ценовые признаки остальных моделей.

In [2]:
def load():
    X = np.load(DATA / 'node_features.npy').astype(np.float32)
    idx = pd.read_csv(DATA / 'node_features_index.csv', parse_dates=['TRADEDATE'])
    nodes = pd.read_csv(DATA / 'graph_nodes.csv')
    idx['node'] = idx['SECID'].map(dict(zip(nodes.SECID, nodes.node_id))).values
    idx['month'] = idx['TRADEDATE'].dt.strftime('%Y-%m')
    ei, ea = np.load(DATA / 'edge_index_corr.npz'), np.load(DATA / 'edge_attr_corr.npz')
    graphs = {}
    for m in ei.files:
        e, E = ei[m], ei[m].shape[1] // 2
        assert (e[:, :E] == e[[1, 0], E:]).all(), 'ожидается раскладка [прямые; обратные]'
        graphs[m] = (torch.tensor(e[:, :E]), torch.tensor(ea[m][:E]))   # храним только прямые дуги
    keep = idx['month'].isin(graphs).values
    idx, X = idx[keep].reset_index(drop=True), X[keep]
    idx['t'] = pd.factorize(idx['TRADEDATE'], sort=True)[0]             # номер торгового дня 0..T−1
    return idx, X, graphs, len(nodes)


def sequences(idx, L=SEQ):
    """Для LSTM: последние L торговых дней бумаги до t включительно, [log-доходность, log(объём / 20-дн. среднее)].
    Окно выровнено по левому краю, его длина возвращается отдельно и маскируется в модели."""
    pr = pd.read_csv(DATA / 'prices_adj.csv', parse_dates=['TRADEDATE']).sort_values(['SECID', 'TRADEDATE'])
    pr['lr'] = np.log1p(pr['RETURN'])
    vma = pr.groupby('SECID')['ADJ_VOLUME'].transform(lambda x: x.rolling(20, min_periods=5).mean())
    pr['lv'] = np.log(pr['ADJ_VOLUME'].clip(lower=1) / vma.clip(lower=1))
    S, lens = np.zeros((len(idx), L, 2), np.float32), np.zeros(len(idx), np.int64)
    for sec, g in pr.groupby('SECID'):
        arr = np.nan_to_num(g[['lr', 'lv']].to_numpy(np.float32))         # пропуск сессии и начало ряда → 0
        rows = np.flatnonzero(idx['SECID'].values == sec)
        pos = np.searchsorted(g['TRADEDATE'].values, idx['TRADEDATE'].values[rows])
        for r, p in zip(rows, pos):
            w = arr[max(0, p - L + 1):p + 1]
            S[r, :len(w)], lens[r] = w, len(w)
    return S, lens


idx, X, graphs, N_NODES = load()
FEAT = X.shape[1]
S, LENS = sequences(idx)
months = sorted(idx['month'].unique())
print(f'строк {len(idx):,}  дней {idx.t.max() + 1}  месяцев {len(months)} ({months[0]} … {months[-1]})  '
      f'графов {len(graphs)}  узлов {N_NODES}  признаков {FEAT}')

строк 33,736  дней 827  месяцев 39 (2022-10 … 2025-12)  графов 40  узлов 46  признаков 775


## Фолды и подготовка признаков

Walk-forward по календарным месяцам. Основная сетка 18/6/6 с шагом 3 даёт четыре полных фолда;
контрольная сетка 24/6/6 с шагом 6 даёт два длинных, у второго тест обрезан до трёх месяцев;
по ней оценивается чувствительность результата к нарезке окон.
Всё, что оценивается по данным (терцильные пороги разметки, скейлеры, веса классов), считается
только на train-части фолда. Последний день train и val выбрасывается: его таргет заглядывает
в первый день следующего сплита. Бумаги, у которых в train фолда меньше 120 строк (новые
тикеры после редомициляции и IPO), не размечаются (`y = −1`): терцилей для них нет. Они остаются
узлами графа, но не входят ни в функцию потерь, ни в метрики.

In [3]:
def make_folds(months, train=18, val=6, test=6, step=3, min_tail=6):
    folds, s = [], 0
    while True:
        tr, va, te = (months[s:s + train], months[s + train:s + train + val],
                      months[s + train + val:s + train + val + test])
        if len(tr) < train or len(va) < min_tail or len(te) < min_tail:
            return folds
        folds.append((tr, va, te)); s += step


def prepare(idx, X, S, lens, fold, purge=1):
    masks = [idx['month'].isin(m).values for m in fold]
    for m in masks[:2]:                                   # purge границы train|val и val|test
        last = np.unique(idx['TRADEDATE'].values[m])[-purge:]
        m &= ~np.isin(idx['TRADEDATE'].values, last)
    tr = masks[0]
    q = idx[tr].groupby('SECID')['FUTURE_RETURN'].quantile([1 / 3, 2 / 3]).unstack()
    fr, lo, hi = idx['FUTURE_RETURN'].values, idx['SECID'].map(q[1 / 3]).values, idx['SECID'].map(q[2 / 3]).values
    y = np.where(fr < lo, 0, np.where(fr > hi, 2, 1))     # 0 DOWN / 1 FLAT / 2 UP по терцилям тикера
    y[idx['SECID'].map(idx[tr].groupby('SECID').size()).fillna(0).values < MIN_TRAIN_ROWS] = -1
    Xs = X.copy()
    Xs[:, PRICE] = StandardScaler().fit(X[tr][:, PRICE]).transform(X[:, PRICE])
    Xs[:, NEWS] = StandardScaler().fit(X[tr & (X[:, FLAG] > 0)][:, NEWS]).transform(X[:, NEWS])
    Xs[X[:, FLAG] == 0, NEWS] = 0                          # нет новостей → нулевой эмбеддинг
    valid = np.arange(S.shape[1]) < lens[:, None]          # паддинг не входит в статистики
    flat = S[tr][valid[tr]]
    Ss = np.where(valid[..., None], (S - flat.mean(0)) / (flat.std(0) + 1e-8), 0).astype(np.float32)
    cnt = np.bincount(y[tr & (y >= 0)], minlength=3)
    return dict(masks=masks, days=[np.unique(idx['t'].values[m]) for m in masks],
                y=y, X=Xs, S=Ss, lens=lens, cw=cnt.sum() / (3 * cnt))


GRIDS = {'4x18': make_folds(months), '2x24': make_folds(months, train=24, step=6, min_tail=3)}
for name, folds in GRIDS.items():
    print(name)
    for i, (tr, va, te) in enumerate(folds):
        print(f'  fold {i}: train {tr[0]}…{tr[-1]}  val {va[0]}…{va[-1]}  test {te[0]}…{te[-1]}')

4x18
  fold 0: train 2022-10…2024-03  val 2024-04…2024-09  test 2024-10…2025-03
  fold 1: train 2023-01…2024-06  val 2024-07…2024-12  test 2025-01…2025-06
  fold 2: train 2023-04…2024-09  val 2024-10…2025-03  test 2025-04…2025-09
  fold 3: train 2023-07…2024-12  val 2025-01…2025-06  test 2025-07…2025-12
2x24
  fold 0: train 2022-10…2024-09  val 2024-10…2025-03  test 2025-04…2025-09
  fold 1: train 2023-04…2025-03  val 2025-04…2025-09  test 2025-10…2025-12


## Батч из нескольких дней

Все 46 тикеров одного дня образуют один граф. Несколько дней складываются в несвязное объединение:
узлы дня *i* сдвигаются на *i* · 46, рёбра берутся из графа соответствующего месяца. Неторгующиеся
в этот день тикеры остаются нулевыми строками, их рёбра выбрасываются, поэтому они не передают
сообщений и в функцию потерь не попадают. Edge dropout выбрасывает ребро целиком (обе дуги),
поэтому после него обратные дуги по-прежнему находятся сдвигом на *E*.

In [4]:
class Days:
    def __init__(self, idx):
        self.rows = [g.index.values for _, g in idx.groupby('t')]    # строки каждого дня
        self.month = idx.groupby('t')['month'].first().values
        self.node = idx['node'].values

    def batch(self, ts, fold, graphs, train=False, p_drop=0.2):
        rows = [self.rows[t] for t in ts]
        off = np.arange(len(ts)) * N_NODES
        pos = np.concatenate([self.node[r] + o for r, o in zip(rows, off)])   # строка → узел батча
        rows = np.concatenate(rows)
        present = torch.zeros(len(ts) * N_NODES, dtype=torch.bool); present[pos] = True
        X = torch.zeros(len(ts) * N_NODES, FEAT)
        X[pos] = torch.from_numpy(fold['X'][rows])
        ei = torch.cat([graphs[self.month[t]][0] + o for t, o in zip(ts, off)], 1)
        ea = torch.cat([graphs[self.month[t]][1] for t in ts])
        keep = present[ei[0]] & present[ei[1]]                                  # рёбра к неторгующимся узлам
        if train:
            keep &= torch.rand(ei.shape[1]) > p_drop
        ei, ea = ei[:, keep], ea[keep]
        E = ei.shape[1]
        ei, ea = torch.cat([ei, ei.flip(0)], 1), torch.cat([ea, ea])            # [прямые; обратные]
        rev = torch.cat([torch.arange(E) + E, torch.arange(E)])                 # индекс обратной дуги
        return dict(X=X, ei=ei, ea=ea, rev=rev, pos=torch.from_numpy(pos), rows=rows,
                    Xr=torch.from_numpy(fold['X'][rows]), S=torch.from_numpy(fold['S'][rows]),
                    len=torch.from_numpy(fold['lens'][rows]), y=torch.from_numpy(fold['y'][rows]))


days = Days(idx)

## Модели

**NSD.** Слой sheaf-диффузии в формулировке Bodnar et al. (2022):
`X ← (1 + tanh ε) X − σ(Δ_F (I ⊗ W₁) X W₂)`, где `Δ_F = D^{-1/2} L_F D^{-1/2}` — нормированный
пучковый лапласиан. На каждую дугу *e = (v, u)* MLP предсказывает карту ограничения общего вида
`F_{v◁e} ∈ R^{d×d}` по `[h_v ‖ h_u ‖ edge_attr]`, карта второго конца `F_{u◁e}` берётся с обратной дуги.
`(L_F X)_v = Σ_e F_{v◁e}ᵀ (F_{v◁e} x_v − F_{u◁e} x_u)`, `D_v = I + Σ_e F_{v◁e}ᵀ F_{v◁e}` (единица
регуляризует нормировку для узлов малой степени). Сам `L_F` положительно полуопределён, но его
внедиагональные блоки `−F_{v◁e}ᵀ F_{u◁e}` знаконеопределены: в отличие от GCN, диффузия может
не только сглаживать соседей, но и «разводить» их, что нужно на отрицательных рёбрах.
`W₁` смешивает размерности стебля, `W₂` — каналы, `ε` обучаемый.

**Бейзлайны.** GCN (вес ребра |ρ|) и GAT (edge_attr `[sign ρ, |ρ|]`) из PyTorch Geometric
на тех же признаках и графе; MLP на тех же 775 признаках без графа; LSTM на 20-дневном окне
цен и объёмов без новостей и без графа; Majority предсказывает мажоритарный класс train.

In [5]:
class SheafLayer(nn.Module):
    def __init__(self, hid, d):
        super().__init__()
        self.d, self.c = d, hid // d                     # hid = d стеблей × c каналов
        self.maps = nn.Sequential(nn.Linear(2 * hid + 2, hid), nn.ELU(), nn.Linear(hid, d * d))
        self.W1, self.W2 = nn.Linear(d, d, bias=False), nn.Linear(self.c, self.c, bias=False)
        self.eps = nn.Parameter(torch.zeros(1))

    def forward(self, h, ei, ea, rev):
        src, dst = ei
        n, d = h.shape[0], self.d
        Fm = self.maps(torch.cat([h[src], h[dst], ea], -1)).view(-1, d, d)            # F_{src◁e}
        D = torch.eye(d).repeat(n, 1, 1).index_add(0, src, Fm.transpose(1, 2) @ Fm)    # I + Σ Fᵀ F
        act = torch.zeros(n, dtype=torch.bool); act[src] = True
        Dis = torch.eye(d).repeat(n, 1, 1)                                             # D^{-1/2}; у изолированных узлов I
        if act.any():
            ev, U = torch.linalg.eigh(D[act])
            Dis[act] = U @ torch.diag_embed(ev.clamp(min=1e-6).rsqrt()) @ U.transpose(1, 2)
        H = h.view(n, d, self.c)
        H = Dis @ self.W2(self.W1(H.transpose(1, 2)).transpose(1, 2))                 # D^{-1/2} (I ⊗ W1) X W2
        msg = Fm.transpose(1, 2) @ (Fm @ H[src] - Fm[rev] @ H[dst])                   # (E, d, c)
        out = Dis @ torch.zeros_like(H).index_add_(0, src, msg)                       # Δ_F (I ⊗ W1) X W2
        return (1 + torch.tanh(self.eps)) * h - F.elu(out.reshape(h.shape))


class NSD(nn.Module):
    def __init__(self, hid=64, d=4, layers=2, drop=0.3):
        super().__init__()
        self.inp, self.out, self.drop = nn.Linear(FEAT, hid), nn.Linear(hid, 3), nn.Dropout(drop)
        self.layers = nn.ModuleList([SheafLayer(hid, d) for _ in range(layers)])

    def forward(self, b):
        h = self.drop(F.elu(self.inp(b['X'])))
        for layer in self.layers:
            h = self.drop(layer(h, b['ei'], b['ea'], b['rev']))
        return self.out(h)[b['pos']]


class GNN(nn.Module):
    def __init__(self, kind, hid=64, layers=2, drop=0.3):
        super().__init__()
        self.kind = kind
        conv = (lambda: GCNConv(hid, hid)) if kind == 'GCN' else (lambda: GATConv(hid, hid // 4, heads=4, edge_dim=2))
        self.inp, self.out, self.drop = nn.Linear(FEAT, hid), nn.Linear(hid, 3), nn.Dropout(drop)
        self.convs = nn.ModuleList([conv() for _ in range(layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(hid) for _ in range(layers)])

    def forward(self, b):
        h = F.elu(self.inp(b['X']))
        for conv, norm in zip(self.convs, self.norms):
            h = conv(h, b['ei'], b['ea'][:, 1]) if self.kind == 'GCN' else conv(h, b['ei'], b['ea'])
            h = self.drop(F.elu(norm(h)))
        return self.out(h)[b['pos']]


class MLP(nn.Module):
    def __init__(self, hid=64, drop=0.3):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(FEAT, hid), nn.ELU(), nn.Dropout(drop),
                                 nn.Linear(hid, hid), nn.ELU(), nn.Dropout(drop), nn.Linear(hid, 3))

    def forward(self, b):
        return self.net(b['Xr'])


class LSTM(nn.Module):
    def __init__(self, hid=64, drop=0.3):
        super().__init__()
        self.lstm, self.drop, self.out = nn.LSTM(2, hid, batch_first=True), nn.Dropout(drop), nn.Linear(hid, 3)

    def forward(self, b):
        packed = nn.utils.rnn.pack_padded_sequence(b['S'], b['len'], batch_first=True, enforce_sorted=False)
        _, (h, _) = self.lstm(packed)                    # h — состояние на последнем непаддинговом шаге
        return self.out(self.drop(h[-1]))


def build(kind, cfg):
    if kind == 'NSD':
        return NSD(cfg.get('hid', 64), cfg.get('d', 4), cfg.get('layers', 2))
    return GNN(kind) if kind in ('GCN', 'GAT') else {'MLP': MLP, 'LSTM': LSTM}[kind]()


for k in ['NSD', 'GCN', 'GAT', 'MLP', 'LSTM']:
    print(f'{k:5s} параметров: {sum(p.numel() for p in build(k, {}).parameters()):,}')

NSD   параметров: 69,253
GCN   параметров: 58,435
GAT   параметров: 59,075
MLP   параметров: 54,019
LSTM  параметров: 17,603


## Обучение

AdamW (lr 10⁻³, weight decay 10⁻⁴), клиппинг градиента 1.0, `ReduceLROnPlateau` по валидационному
MCC, early stopping с patience 10, не более 100 эпох. Функция потерь: кросс-энтропия с весами,
обратными частотам классов на train; неразмеченные строки (`y = −1`) игнорируются. Порядок дней
перемешивается каждую эпоху.

In [6]:
def metrics(y, p):
    return dict(acc=accuracy_score(y, p), mcc=matthews_corrcoef(y, p),
                f1=f1_score(y, p, average='macro'), f1_down=f1_score(y, p, labels=[0], average='macro'))


@torch.no_grad()
def predict(model, ts, fold, graphs):
    """Вероятности классов для размеченных строк дней ts."""
    model.eval()
    out, rows = [], []
    for i in range(0, len(ts), K_DAYS):
        b = days.batch(ts[i:i + K_DAYS], fold, graphs)
        out.append(torch.softmax(model(b), -1)); rows.append(b['rows'])
    p, rows = torch.cat(out).numpy(), np.concatenate(rows)
    ok = fold['y'][rows] >= 0
    return p[ok], rows[ok]


def train(kind, cfg, fold, graphs, seed, epochs=100, patience=10):
    seed_all(seed)
    model = build(kind, cfg)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, 'max', factor=0.5, patience=5, min_lr=1e-5)
    crit = nn.CrossEntropyLoss(weight=torch.tensor(fold['cw'], dtype=torch.float32), ignore_index=-1)
    tr, va, te = fold['days']
    best, state, wait, t0 = -np.inf, None, 0, time.time()
    for ep in range(1, epochs + 1):
        model.train()
        order = np.random.permutation(tr)
        for i in range(0, len(order), K_DAYS):
            b = days.batch(order[i:i + K_DAYS], fold, graphs, train=True)
            if not (b['y'] >= 0).any():
                continue
            loss = crit(model(b), b['y'])
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        p, rows = predict(model, va, fold, graphs)
        score = matthews_corrcoef(fold['y'][rows], p.argmax(1))
        sched.step(score)
        if score > best + 1e-9:
            best, state, wait = score, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            wait += 1
            if wait >= patience:
                break
    model.load_state_dict(state)
    p, rows = predict(model, te, fold, graphs)
    return p, rows, dict(best_val=best, epochs=ep, sec=round(time.time() - t0, 1))

## Эксперименты

Шесть моделей на основной сетке; абляции NSD: без новостей, на статическом секторном графе,
по размерности стебля *d* (2, 4, 8 при одной и той же ширине 64, чтобы не смешивать d с числом
параметров) и числу слоёв *L*; MLP без новостей как вторая проверка H2; отдельно NSD на контрольной
сетке 24/6/6. Сид зависит от фолда и номера повтора (`42 + 10·fold + seed`), поэтому любой прогон
воспроизводится в отдельности.

In [7]:
es = np.load(DATA / 'edge_index_sector.npy'); E = es.shape[1] // 2
SECTOR = {m: (torch.tensor(es[:, :E]), torch.tensor(np.tile([0., 1.], (E, 1)), dtype=torch.float32))
          for m in graphs}                                   # статический граф, атрибут [0, 1]

def news_off(fold):
    f = dict(fold); f['X'] = fold['X'].copy(); f['X'][:, FLAG] = 0; f['X'][:, NEWS] = 0
    return f

EXPERIMENTS = {   # тег: (модель, отличия от базовой конфигурации NSD: hid 64, d 4, L 2)
    'NSD': ('NSD', {}), 'GCN': ('GCN', {}), 'GAT': ('GAT', {}),
    'MLP': ('MLP', {}), 'LSTM': ('LSTM', {}), 'Majority': ('Majority', {}),
    'NSD news_off': ('NSD', {'news_off': True}), 'MLP news_off': ('MLP', {'news_off': True}),
    'NSD sector': ('NSD', {'graph': 'sector'}),
    'NSD d=2': ('NSD', {'d': 2}), 'NSD d=8': ('NSD', {'d': 8}),
    'NSD L=1': ('NSD', {'layers': 1}), 'NSD L=3': ('NSD', {'layers': 3}),
}
PLAN = [(tag, '4x18') for tag in EXPERIMENTS] + [('NSD', '2x24')]

runs_path = RESULTS / 'runs.csv'
done = set(map(tuple, pd.read_csv(runs_path)[['tag', 'grid', 'fold', 'seed']].itertuples(index=False))) \
       if runs_path.exists() else set()
prepared = {}
for tag, grid in PLAN:
    kind, cfg = EXPERIMENTS[tag]
    pred_path = RESULTS / 'predictions' / f"{tag.replace(' ', '_').replace('=', '')}_{grid}.csv.gz"
    t0 = time.time()
    for f, spec in enumerate(GRIDS[grid]):
        if (grid, f) not in prepared:
            prepared[grid, f] = prepare(idx, X, S, LENS, spec)
        fold = news_off(prepared[grid, f]) if cfg.get('news_off') else prepared[grid, f]
        G = SECTOR if cfg.get('graph') == 'sector' else graphs
        for s in range(SEEDS):
            if (tag, grid, f, s) in done:
                continue
            if kind == 'Majority':
                rows = np.flatnonzero(fold['masks'][2] & (fold['y'] >= 0))
                tr_y = fold['y'][fold['masks'][0]]
                p = np.zeros((len(rows), 3), np.float32); p[:, np.bincount(tr_y[tr_y >= 0]).argmax()] = 1
                info = dict(best_val=np.nan, epochs=0, sec=0.0)
            else:
                p, rows, info = train(kind, cfg, fold, G, seed=42 + 10 * f + s)
            m = metrics(fold['y'][rows], p.argmax(1))
            # сначала предсказания, потом строка в runs.csv: прерванный прогон не считается сделанным
            pd.DataFrame(dict(tag=tag, grid=grid, fold=f, seed=s, SECID=idx['SECID'].values[rows],
                              TRADEDATE=idx['TRADEDATE'].values[rows], FUTURE_RETURN=idx['FUTURE_RETURN'].values[rows],
                              y=fold['y'][rows], p_down=p[:, 0], p_flat=p[:, 1], p_up=p[:, 2])).to_csv(
                pred_path, mode='a', header=not pred_path.exists(), index=False, float_format='%.5g')
            pd.DataFrame([dict(tag=tag, grid=grid, fold=f, seed=s, **m, **info)]).to_csv(
                runs_path, mode='a', header=not runs_path.exists(), index=False)
    r = pd.read_csv(runs_path).query('tag == @tag and grid == @grid')
    by_fold = r.groupby('fold')['mcc'].median()
    print(f'{tag:13s} {grid}  MCC по фолдам {" ".join(f"{v:+.3f}" for v in by_fold)}  '
          f'среднее {by_fold.mean():+.4f}  ({time.time() - t0:.0f} с)')

NSD           4x18  MCC по фолдам +0.003 +0.017 +0.013 -0.023  среднее +0.0023  (264 с)


GCN           4x18  MCC по фолдам -0.019 -0.022 +0.002 -0.028  среднее -0.0167  (108 с)


GAT           4x18  MCC по фолдам -0.012 -0.006 +0.027 -0.021  среднее -0.0030  (155 с)


MLP           4x18  MCC по фолдам +0.000 -0.005 +0.015 +0.008  среднее +0.0043  (55 с)


LSTM          4x18  MCC по фолдам +0.070 +0.063 +0.066 +0.072  среднее +0.0675  (790 с)


...

NSD d=2       4x18  MCC по фолдам +0.005 +0.002 +0.024 -0.007  среднее +0.0058  (220 с)


NSD d=8       4x18  MCC по фолдам +0.005 +0.011 +0.013 -0.023  среднее +0.0017  (307 с)


NSD L=1       4x18  MCC по фолдам +0.007 +0.015 +0.019 -0.016  среднее +0.0062  (172 с)


NSD L=3       4x18  MCC по фолдам +0.003 -0.007 +0.014 -0.018  среднее -0.0019  (354 с)


NSD           2x24  MCC по фолдам +0.040 +0.036  среднее +0.0379  (232 с)


In [8]:
runs = pd.read_csv(runs_path)
tab = runs.groupby(['grid', 'tag', 'fold'])['mcc'].median().unstack('fold')
tab['mean'], tab['sd'] = tab.mean(1), tab.std(1)
tab.round(4).sort_values(['grid', 'mean'], ascending=[False, False])

fold                    0       1       2       3    mean      sd
grid tag                                                         
4x18 LSTM          0.0701  0.0628  0.0656  0.0715  0.0675  0.0040
     MLP news_off  0.0513  0.0561  0.0143  0.0058  0.0319  0.0255
     NSD news_off  0.0460  0.0566  0.0153  0.0086  0.0316  0.0233
     NSD L=1       0.0068  0.0153  0.0187 -0.0161  0.0062  0.0157
     NSD d=2       0.0053  0.0016  0.0237 -0.0073  0.0058  0.0130
     NSD sector    0.0065  0.0136  0.0178 -0.0168  0.0053  0.0154
     MLP           0.0004 -0.0055  0.0149  0.0075  0.0043  0.0088
     NSD           0.0026  0.0171  0.0127 -0.0232  0.0023  0.0180
     NSD d=8       0.0053  0.0115  0.0128 -0.0228  0.0017  0.0166
     Majority      0.0000  0.0000  0.0000  0.0000  0.0000  0.0000
     NSD L=3       0.0033 -0.0072  0.0141 -0.0178 -0.0019  0.0137
     GAT          -0.0120 -0.0058  0.0272 -0.0213 -0.0030  0.0211
     GCN          -0.0189 -0.0215  0.0020 -0.0282 -0.0167  0.0130
2x24 NSD           0.0399  0.0358     NaN     NaN  0.0379  0.0029